# המרת DictaLM-3.0-1.7B-Instruct ל-GGUF (Kaggle, ללא GPU)

ה-**Instruct** (לא-Thinking) הוא המועמד הנכון לחברותא: מומחה עברית, עונה ישר,
בלי שרשראות חשיבה. אין לו GGUF רשמי — אז מייצרים בעצמנו (ארכיטקטורת Qwen3, נתמכת מלא).

⚙️ **הגדרות:** Accelerator = **None (CPU)** מספיק · **Internet = On**

⏱️ **זמן צפוי:** ~20-35 דק' (הורדה ~3.4GB → המרה → קוונטיזציה → Q8_0 ‎~1.8GB)


### 1. התקנות + llama.cpp (כלי ההמרה)


In [ ]:
!pip install -q -U huggingface_hub sentencepiece
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt


### 2. בניית llama-quantize (CPU, כמה דקות)


In [ ]:
!cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /kaggle/working/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!ls /kaggle/working/llama.cpp/build/bin/llama-quantize


### 3. הורדת המשקלים המקוריים (~3.4GB bf16)


In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    repo_id="dicta-il/DictaLM-3.0-1.7B-Instruct",
    local_dir="/kaggle/tmp/dictalm-instruct",
)
print("downloaded to:", model_dir)


### 4. המרה ל-GGUF (F16 ביניים)


In [ ]:
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/tmp/dictalm-instruct --outfile /kaggle/tmp/DictaLM-3.0-1.7B-Instruct-F16.gguf --outtype f16
!ls -lh /kaggle/tmp/*.gguf


### 5. קוונטיזציה ל-Q8_0 (האיזון הנכון למודל קטן)


In [ ]:
!/kaggle/working/llama.cpp/build/bin/llama-quantize /kaggle/tmp/DictaLM-3.0-1.7B-Instruct-F16.gguf /kaggle/working/DictaLM-3.0-1.7B-Instruct-Q8_0.gguf Q8_0
!ls -lh /kaggle/working/*.gguf


### 6. (אופציונלי) גם Q4_K_M קליל יותר (~1.1GB)


In [ ]:
!/kaggle/working/llama.cpp/build/bin/llama-quantize /kaggle/tmp/DictaLM-3.0-1.7B-Instruct-F16.gguf /kaggle/working/DictaLM-3.0-1.7B-Instruct-Q4_K_M.gguf Q4_K_M
!ls -lh /kaggle/working/*.gguf


### 7. הורדה ושימוש במחשב

הורד את `DictaLM-3.0-1.7B-Instruct-Q8_0.gguf` מ-Output (צד ימין), שים בתיקיית
הפרויקט תחת `models\`, וצור מודל ב-Ollama:

```powershell
# Modelfile כבר מוכן בפרויקט: models\Modelfile.dictalm-instruct
ollama create dictalm3-instruct -f models\Modelfile.dictalm-instruct

# ואז המערכת עוברת אליו ב-config בלבד:
$env:CHAVRUTA_LLM_MODEL = "dictalm3-instruct"
python scripts/ask.py 'מה אומר רש"י על בריאת האור?'
```
